# Project 13 - H&M Personalized Fashion Recommendations - EDA

**Author:** Sandeep Grover, Liora MLE Programme, Cohort 6973

This notebook is the exploratory pass over the Kaggle H&M Personalized Fashion Recommendations dataset. The dataset is roughly 30 GB once extracted (transactions, customers, articles, plus a 26 GB image folder). Phase 1 of this project documents the schema and checks the loading skeleton without downloading the data; cells below are wired but not executed.

## Goals

1. Load the three tabular CSVs and confirm shapes, dtypes, missingness.
2. Characterise the implicit-feedback target: transactions per customer, transactions per article, sparsity.
3. Explore recency and seasonality structure: weekly volume, day-of-week, time-since-last-purchase.
4. Examine article-side metadata: product type, garment group, colour distribution, price by section.
5. Examine customer-side metadata: age distribution, club membership, fashion-news subscription.
6. Stage notes for the baseline ALS model and the two-tower advanced model.

## 1. Imports and configuration

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('../data')
TRANSACTIONS_CSV = DATA_DIR / 'transactions_train.csv'
CUSTOMERS_CSV = DATA_DIR / 'customers.csv'
ARTICLES_CSV = DATA_DIR / 'articles.csv'

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

print('Data dir exists:', DATA_DIR.exists())
for p in [TRANSACTIONS_CSV, CUSTOMERS_CSV, ARTICLES_CSV]:
    print(p.name, 'exists:', p.exists())

## 2. Load the three tabular files

`transactions_train.csv` is roughly 31.8 million rows. On a 32 GB machine it loads in about 90 s. For Phase 1 we also expose a `nrows`-limited path so the notebook can be re-run cheaply during development.

In [ ]:
DEV_MODE = True
DEV_NROWS = 2_000_000  # set None for full file

txn_dtypes = {
    'customer_id': 'string',
    'article_id': 'int64',
    'price': 'float32',
    'sales_channel_id': 'int8',
}

transactions = pd.read_csv(
    TRANSACTIONS_CSV,
    dtype=txn_dtypes,
    parse_dates=['t_dat'],
    nrows=DEV_NROWS if DEV_MODE else None,
)
customers = pd.read_csv(CUSTOMERS_CSV)
articles = pd.read_csv(ARTICLES_CSV)

print('transactions:', transactions.shape)
print('customers   :', customers.shape)
print('articles    :', articles.shape)

## 3. Schema, dtypes, head

Inspect each table's schema. The transactions table is the implicit-feedback signal; customers and articles supply tower features for the two-tower model.

In [ ]:
transactions.head()

In [ ]:
transactions.dtypes

In [ ]:
customers.head()

In [ ]:
customers.dtypes

In [ ]:
articles.head()

In [ ]:
articles.dtypes

## 4. Describe and missingness

We expect:
- `transactions`: zero target missingness (the table is the target), `price` may have a thin missing tail on returned items.
- `customers`: `age` has a meaningful missing share (roughly 15%), `FN` and `Active` are mostly missing because they only flip when the customer opts in, `club_member_status` and `fashion_news_frequency` are categorical with explicit `NONE` values.
- `articles`: `detail_desc` has a small missing share (a few percent).

In [ ]:
transactions.describe(include='all', datetime_is_numeric=True)

In [ ]:
for name, df in [('transactions', transactions), ('customers', customers), ('articles', articles)]:
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    print(f'== {name} missing ==')
    print(miss)
    print()

## 5. Implicit-feedback target shape

How sparse is the customer-article interaction matrix, and how skewed are the per-customer and per-article transaction counts? These two distributions determine whether ALS with a single global confidence parameter is reasonable, or whether we need a recency-weighted confidence.

In [ ]:
n_customers = transactions['customer_id'].nunique()
n_articles = transactions['article_id'].nunique()
n_interactions = len(transactions)
density = n_interactions / (n_customers * n_articles)
print(f'unique customers: {n_customers:,}')
print(f'unique articles : {n_articles:,}')
print(f'interactions    : {n_interactions:,}')
print(f'density         : {density:.6%}')

In [ ]:
txn_per_customer = transactions.groupby('customer_id').size()
txn_per_article = transactions.groupby('article_id').size()

print('per-customer:', txn_per_customer.describe())
print()
print('per-article :', txn_per_article.describe())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
txn_per_customer.clip(upper=200).hist(bins=80, ax=ax[0])
ax[0].set_title('Transactions per customer (clipped at 200)')
ax[0].set_yscale('log')
txn_per_article.clip(upper=2000).hist(bins=80, ax=ax[1])
ax[1].set_title('Transactions per article (clipped at 2000)')
ax[1].set_yscale('log')
plt.tight_layout()

## 6. Temporal structure: recency and seasonality

Fashion is brutally seasonal. We plot weekly transaction volume across the full window, and per day-of-week, to confirm that:

1. The volume trend has a clear seasonal rhythm (summer peaks, winter sales bumps).
2. Weekend volume dominates online traffic.
3. The last 4 to 8 weeks of training data carry the freshest signal for the held-out final week.

These observations justify a recency decay in the ALS confidence weight and a recency-weighted sampler in the two-tower model.

In [ ]:
weekly = transactions.set_index('t_dat').resample('W').size()
fig, ax = plt.subplots(figsize=(11, 3.5))
weekly.plot(ax=ax)
ax.set_title('Weekly transaction volume')
ax.set_ylabel('transactions / week')

In [ ]:
dow = transactions['t_dat'].dt.day_name().value_counts().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
)
dow.plot.bar(figsize=(7, 3.5), title='Transactions by day of week')

## 7. Article side: product mix, colour, price

The two-tower advanced model consumes article metadata as the item tower. We characterise the marginal distributions of the categorical metadata columns and the price tail to decide:

1. Which categorical columns are worth embedding versus one-hot.
2. Whether `prod_name` is high-enough cardinality to warrant a small text-tower (TF-IDF or fastText) instead of a categorical embedding.
3. How to bucket price for the user-tower's average-spend feature.

In [ ]:
for col in ['product_group_name', 'garment_group_name', 'index_group_name', 'index_name', 'section_name']:
    print(f'== {col} ({articles[col].nunique()} unique) ==')
    print(articles[col].value_counts().head(10))
    print()

In [ ]:
transactions['price'].describe(percentiles=[0.05, 0.5, 0.95, 0.99])

## 8. Customer side: age, membership, news

User-tower features. We expect `age` to be the dominant numeric, `club_member_status` and `fashion_news_frequency` to be the dominant categoricals, and a small `FN` / `Active` flag pair.

In [ ]:
customers['age'].describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
customers['age'].clip(upper=90).hist(bins=40, ax=ax[0])
ax[0].set_title('Customer age')
customers['club_member_status'].fillna('NaN').value_counts().plot.bar(ax=ax[1])
ax[1].set_title('club_member_status')
plt.tight_layout()

## 9. Hold-out and evaluation framing

The Kaggle metric is MAP@12 on the final week's purchases. We mirror this in offline evaluation:

- Train: transactions strictly before the cut (everything up to 2020-09-15).
- Hold-out: transactions in the final week (2020-09-16 to 2020-09-22).
- For each customer with at least one hold-out purchase, score top-12 candidates and compute average precision; mean over customers gives MAP@12.

This split mirrors the public-test framing, lets us tune ALS regularisation and two-tower hyperparameters offline, and gives a calibrated estimate of competition leaderboard score before any submission is made.

In [ ]:
CUT_DATE = pd.Timestamp('2020-09-16')
train_part = transactions[transactions['t_dat'] < CUT_DATE]
holdout = transactions[transactions['t_dat'] >= CUT_DATE]
print('train rows  :', len(train_part))
print('holdout rows:', len(holdout))
print('holdout customers:', holdout['customer_id'].nunique())

## 10. Next steps

1. Build the customer-by-article CSR sparse matrix with recency-weighted confidence and fit `implicit.als.AlternatingLeastSquares` (`src/model_baseline.py`).
2. Stage article and customer feature tables for the two-tower model; cache article metadata as Parquet (`articles.parquet`).
3. Train the two-tower neural recommender with sampled-softmax over in-batch plus mixed negatives (`src/model_advanced.py`).
4. Index item embeddings in Faiss HNSW for sub-50 ms retrieval at inference.
5. Evaluate MAP@12 / Recall@12 on the final-week hold-out and submit.